# Code Review Agent — llama3.2 (Ollama)

The user provides a file path in natural language.  
The agent reads the file and outputs structured code review comments.

**Prerequisites:** Ollama must be running (`ollama serve`) with `llama3.2` pulled.

## 1. Install dependencies (run once)

In [1]:
# !pip install langchain-ollama langchain-core langgraph

## 2. LLM — llama3.2 via Ollama

In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

## 3. Tools

In [3]:
from langchain_core.tools import tool


@tool
def read_file(path: str) -> str:
    """Read the complete contents of a source file given its path."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"ERROR: File not found — {path}"
    except Exception as e:
        return f"ERROR reading file: {e}"


@tool
def list_files(path: str = ".") -> str:
    """List all files recursively under a directory. Useful when the user gives a folder instead of a file."""
    from pathlib import Path
    try:
        files = [str(p) for p in Path(path).rglob("*") if p.is_file()]
        return "\n".join(files) if files else "No files found."
    except Exception as e:
        return f"ERROR listing files: {e}"


tools = [read_file, list_files]

## 4. Code review agent

In [4]:
from langgraph.prebuilt import create_react_agent

SYSTEM_PROMPT = """\
You are an expert code reviewer. When the user provides a file path, you must:

1. Use the `read_file` tool to read the file contents.
2. Analyse the code thoroughly.
3. Produce a structured code review with the following sections:

   ### Summary
   Brief description of what the code does.

   ### Issues
   List every problem found. For each issue include:
   - Severity: Critical / Major / Minor / Info
   - Line(s): approximate line number(s) if identifiable
   - Description: clear explanation of the problem
   - Suggestion: how to fix it

   ### Positive Aspects
   What the code does well.

   ### Recommendations
   Overall improvement suggestions (architecture, naming, testing, etc.).

Be specific and reference actual code snippets where relevant.
If the file cannot be read, report the error clearly.
"""

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT
)

print("Code review agent ready")

Code review agent ready


/var/folders/9v/l_th953n4270qgzv2_7ygdd80000gn/T/ipykernel_88271/1703071899.py:30: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


## 5. Run a code review

Set `file_to_review` to any file path you want reviewed.

In [5]:
# ── Change this to the file you want reviewed ─────────────────────────────
file_to_review = "/Users/zainabfirdaus/git/learn/python3/ai/agent_flow.py"

user_prompt = f"Please review the file: {file_to_review}"

result = agent.invoke(
    {
        "messages": [
            ("user", user_prompt)
        ]
    }
)

print(result["messages"][-1].content)

### Summary
The provided file `agent_flow.py` appears to be a part of the LangChain project, which is an open-source framework for building conversational AI models. The script initializes an agent with a zero-shot react description and runs it on a simple task to retrieve the current weather in New York.

### Issues

*   **Severity:** Critical
*   **Line(s):** 1-5
*   **Description:** The `get_current_weather` function is not properly defined. It should take two parameters: `location` and `weather_data`. However, it only takes one parameter `location`.
*   **Suggestion:** Define the `get_current_weather` function with two parameters to correctly retrieve the current weather in a given location.

    ```python
def get_current_weather(location: str, weather_data):
    return f"The current weather in {location} is 20 degrees Celsius with clear skies."
```

*   **Severity:** Major
*   **Line(s):** 10-12
*   **Description:** The `initialize_agent` function does not handle potential excepti

## 6. Interactive — review any file you type

In [6]:
file_path = input("Enter the file path to review: ").strip()

result = agent.invoke(
    {
        "messages": [
            ("user", f"Please review the file: {file_path}")
        ]
    }
)

print(result["messages"][-1].content)

Enter the file path to review:  /Users/zainabfirdaus/git/learn/python3/ai/agent_flow.py


### Summary
The provided file `agent_flow.py` appears to be a part of the LangChain project, which is an open-source framework for building conversational AI models. The script initializes an agent with a zero-shot react description and runs it on a simple task to retrieve the current weather in New York.

### Issues

*   **Severity:** Critical
*   **Line(s):** 1-5
*   **Description:** The `get_current_weather` function is not properly defined. It should take two parameters: `location` and `weather_data`. However, it only takes one parameter `location`.
*   **Suggestion:** Define the `get_current_weather` function with two parameters to correctly retrieve the current weather in a given location.

    ```python
def get_current_weather(location: str, weather_data):
    return f"The current weather in {location} is 20 degrees Celsius with clear skies."
```

*   **Severity:** Major
*   **Line(s):** 10-12
*   **Description:** The `initialize_agent` function does not handle potential excepti

## 7. Review multiple files in a loop

In [ ]:
files_to_review = [
    "/Users/zainabfirdaus/git/learn/python3/notebooks/rag/RAG_generation_OracleDB_ollama.ipynb",
    # Add more file paths here
]

for fp in files_to_review:
    print("\n" + "=" * 70)
    print(f"REVIEWING: {fp}")
    print("=" * 70)

    res = agent.invoke(
        {
            "messages": [
                ("user", f"Please review the file: {fp}")
            ]
        }
    )

    print(res["messages"][-1].content)